In [2]:
from __future__ import annotations
import gc, json, os, random, shutil
from dataclasses import dataclass
from typing import Dict, List

import numpy as np, tensorflow as tf
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

def stage(name: str): print(f"\n{'=' * 100}\n{name}\n{'=' * 100}")
def uniq(xs: List[str]) -> List[str]:
    seen, out = set(), []
    for x in xs:
        if x not in seen: seen.add(x); out.append(x)
    return out
def mkdirs(*paths: str): [os.makedirs(p, exist_ok=True) for p in paths]
def seed_all(seed: int): random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def cleanup(spark: SparkSession | None = None, *objs: object, clear_tf: bool = False):
    for o in objs:
        try: o.unpersist(blocking=True)
        except Exception: pass
    if spark is not None:
        for fn in (spark.catalog.clearCache, lambda: spark._jvm.java.lang.System.gc()):
            try: fn()
            except Exception: pass
    if clear_tf:
        try: tf.keras.backend.clear_session()
        except Exception: pass
    gc.collect()

@dataclass
class Config:
    input_path: str = "/user/data/kshape/feature_engineering"
    hdfs_work_dir: str = "/user/data/kshape/model"
    local_out_dir: str = "/workspace/code/kshape/model"
    tf_connector_jar: str = "/workspace/code/kshape/lib/spark-tensorflow-connector_2.12-1.11.0.jar"
    tfrecord_jar: str = "/workspace/code/kshape/lib/spark-tfrecord_2.12-0.7.0.jar"
    time_col: str = "pickup_bin_30m"
    loc_col: str = "PULocationID"
    target_col: str = "pickup_demand"
    split_col: str = "dataset_split"
    time_key_col: str = "time_key"
    tabular_features: tuple[str, ...] = ("hour", "minute", "day_of_week", "demand_t_1", "rolling_mean_24h", "cluster_id", "intra_cluster_similarity")
    sequence_features: tuple[str, ...] = ("pickup_demand", "ewma_output", "rolling_mean_24h", "day_of_week")
    split_aliases_to_validation: tuple[str, ...] = ("val", "valid", "validation")
    valid_splits: tuple[str, ...] = ("train", "validation", "test")
    seq_window: int = 48
    xgb_num_workers: int = 2
    random_state: int = 42
    def local(self, *parts: str) -> str: return os.path.join(self.local_out_dir, *parts)
    def hdfs(self, *parts: str) -> str: return "/".join([self.hdfs_work_dir.rstrip("/"), *parts])
    @property
    def spark_jars(self) -> str: return ",".join((self.tf_connector_jar, self.tfrecord_jar))
    @property
    def spark_cp(self) -> str: return ":".join((self.tf_connector_jar, self.tfrecord_jar))

@dataclass
class SequenceScalerStats:
    seq_mean: np.ndarray
    seq_std: np.ndarray
    y_min: float
    y_denom: float
    def to_dict(self) -> Dict[str, object]:
        return {"seq_mean": self.seq_mean.tolist(), "seq_std": self.seq_std.tolist(), "y_min": float(self.y_min), "y_denom": float(self.y_denom)}

def build_spark(c: Config) -> SparkSession:
    stage("SPARK RUNTIME"); print("[SPARK JARS]", c.spark_jars)
    return (SparkSession.builder.appName("Distributed-Forecast-Prepare")
            .config("spark.jars", c.spark_jars)
            .config("spark.driver.extraClassPath", c.spark_cp)
            .config("spark.executor.extraClassPath", c.spark_cp)
            .config("spark.master", "yarn")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", "192.168.1.111")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .getOrCreate())

class DataProcessor:
    def __init__(self, spark: SparkSession, c: Config): self.spark, self.c = spark, c

    def run(self) -> SequenceScalerStats:
        stage("1/2 - PREPARE DATA")
        raw = clean = seq = None
        try:
            raw = self.spark.read.parquet(self.c.input_path)
            self._validate(raw)
            clean = self._clean(raw)
            self._write_tabular(clean)
            seq = self._build_sequence(clean)
            scaler = self._compute_scaler(seq)
            self._write_sequence_parquet(seq)
            self._write_tfrecord(seq)
            self._save_scaler(scaler)
            return scaler
        finally:
            cleanup(self.spark, seq, clean, raw)

    def _validate(self, df: DataFrame):
        need = {self.c.time_col, self.c.loc_col, self.c.target_col, self.c.split_col, *self.c.tabular_features, *self.c.sequence_features}
        missing = sorted(need - set(df.columns))
        if missing: raise ValueError(f"Input parquet thiếu cột bắt buộc: {missing}")

    def _clean(self, df: DataFrame) -> DataFrame:
        cols = uniq([self.c.time_col, self.c.loc_col, self.c.target_col, self.c.split_col, *self.c.tabular_features, *self.c.sequence_features])
        return (df.select(*cols)
                .withColumn(self.c.split_col, F.lower(F.trim(F.col(self.c.split_col))))
                .withColumn(self.c.split_col, F.when(F.col(self.c.split_col).isin(*self.c.split_aliases_to_validation), "validation").otherwise(F.col(self.c.split_col)))
                .dropna(subset=[self.c.time_col, self.c.loc_col, self.c.target_col, self.c.split_col])
                .filter(F.col(self.c.split_col).isin(*self.c.valid_splits))
                .withColumn(self.c.time_key_col, F.date_format(F.col(self.c.time_col), "yyyy-MM-dd HH:mm:ss"))
                .repartition(max(8, self.c.xgb_num_workers * 4), self.c.loc_col)
                .persist(StorageLevel.MEMORY_AND_DISK))

    def _write_tabular(self, clean: DataFrame):
        stage("WRITE TABULAR PARQUET")
        tab = None
        try:
            base, feats = [self.c.time_col, self.c.time_key_col, self.c.loc_col, self.c.split_col, self.c.target_col], list(self.c.tabular_features)
            tab = clean.select(*(base + feats)).fillna(0.0, subset=feats)
            tab = VectorAssembler(inputCols=feats, outputCol="features_vector", handleInvalid="keep").transform(tab).select(*base, "features_vector")
            tab.write.mode("overwrite").partitionBy(self.c.split_col).parquet(self.c.hdfs("prepared", "tabular"))
        finally:
            cleanup(self.spark, tab)

    def _build_sequence(self, clean: DataFrame) -> DataFrame:
        stage("BUILD SEQUENCE WINDOWS")
        cols = uniq([self.c.time_col, self.c.time_key_col, self.c.loc_col, self.c.split_col, self.c.target_col, *self.c.sequence_features])
        w = Window.partitionBy(self.c.loc_col).orderBy(self.c.time_col).rowsBetween(-self.c.seq_window + 1, 0)
        return (clean.select(*cols)
                .fillna(0.0, subset=list(self.c.sequence_features))
                .withColumn("_step", F.array(*[F.col(x).cast("float") for x in self.c.sequence_features]))
                .withColumn("_pair", F.struct(F.col(self.c.time_col).alias("ts"), F.col("_step").alias("vals")))
                .withColumn("_window", F.collect_list("_pair").over(w))
                .withColumn("sequence_array", F.expr("transform(array_sort(_window), x -> x.vals)"))
                .filter(F.size("sequence_array") == self.c.seq_window)
                .select(self.c.time_col, self.c.time_key_col, self.c.loc_col, self.c.split_col, self.c.target_col, "sequence_array")
                .persist(StorageLevel.MEMORY_AND_DISK))

    def _compute_scaler(self, seq: DataFrame) -> SequenceScalerStats:
        train_steps = None
        try:
            train_steps = seq.filter(F.col(self.c.split_col) == "train").select(F.explode("sequence_array").alias("step"))
            exprs = [e for i in range(len(self.c.sequence_features)) for e in (F.avg(F.col("step")[i]).alias(f"m{i}"), F.stddev_pop(F.col("step")[i]).alias(f"s{i}"))]
            stats = train_steps.agg(*exprs).first()
            y_stats = seq.filter(F.col(self.c.split_col) == "train").agg(F.min(self.c.target_col).alias("mn"), F.max(self.c.target_col).alias("mx")).first()
            mean = np.array([float(stats[f"m{i}"] or 0.0) for i in range(len(self.c.sequence_features))], np.float32)
            std = np.array([1.0 if stats[f"s{i}"] is None or float(stats[f"s{i}"]) < 1e-6 else float(stats[f"s{i}"]) for i in range(len(self.c.sequence_features))], np.float32)
            return SequenceScalerStats(mean, std, float(y_stats["mn"]), max(float(y_stats["mx"]) - float(y_stats["mn"]), 1e-6))
        finally:
            cleanup(self.spark, train_steps)

    def _write_sequence_parquet(self, seq: DataFrame):
        flat = None
        try:
            flat = seq.withColumn("sequence_flat", F.flatten("sequence_array")).select(self.c.time_col, self.c.time_key_col, self.c.loc_col, self.c.split_col, self.c.target_col, "sequence_flat")
            flat.write.mode("overwrite").partitionBy(self.c.split_col).parquet(self.c.hdfs("prepared", "sequence"))
        finally:
            cleanup(self.spark, flat)

    def _write_tfrecord(self, seq: DataFrame):
        stage("WRITE TFRECORD")
        tfdf = part = None
        try:
            tfdf = seq.withColumn("sequence_flat", F.flatten("sequence_array")).select(
                F.col(self.c.loc_col).cast("long").alias(self.c.loc_col),
                F.col(self.c.time_key_col).cast("string").alias(self.c.time_key_col),
                F.col(self.c.split_col).cast("string").alias(self.c.split_col),
                F.col(self.c.target_col).cast("float").alias(self.c.target_col),
                F.expr("transform(sequence_flat, x -> cast(x as float))").alias("sequence_flat"),
            )
            for split in self.c.valid_splits:
                part = tfdf.filter(F.col(self.c.split_col) == split)
                part.repartition(max(8, self.c.xgb_num_workers * 4), self.c.loc_col).write.format("tfrecord").option("recordType", "Example").mode("overwrite").save(self.c.hdfs("prepared", "tfrecords", split))
                cleanup(self.spark, part); part = None
        finally:
            cleanup(self.spark, tfdf, part)

    def _save_scaler(self, scaler: SequenceScalerStats):
        with open(self.c.local("sequence_scaler_stats.json"), "w", encoding="utf-8") as f:
            json.dump(scaler.to_dict(), f, indent=2)

class PipelineRunner:
    def __init__(self, c: Config):
        self.c = c
        mkdirs(c.local_out_dir)
        seed_all(c.random_state)
        self.spark = build_spark(c)
        self.spark.sparkContext.setLogLevel("WARN")

    def run(self):
        DataProcessor(self.spark, self.c).run()
        print("\n[PART 1 COMPLETE] prepared/tabular, prepared/sequence, prepared/tfrecords, sequence_scaler_stats.json")

if __name__ == "__main__":
    runner = None
    try:
        runner = PipelineRunner(Config())
        runner.run()
    finally:
        del runner
        gc.collect()



SPARK RUNTIME
[SPARK JARS] /workspace/code/kshape/lib/spark-tensorflow-connector_2.12-1.11.0.jar,/workspace/code/kshape/lib/spark-tfrecord_2.12-0.7.0.jar


26/05/04 00:19:55 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.Con


1/2 - PREPARE DATA



WRITE TABULAR PARQUET



BUILD SEQUENCE WINDOWS



WRITE TFRECORD


Py4JJavaError: An error occurred while calling o403.save.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: tfrecord. Please find packages at `https://spark.apache.org/third-party-projects.html`.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:724)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:647)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:697)
	at org.apache.spark.sql.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:863)
	at org.apache.spark.sql.DataFrameWriter.saveInternal(DataFrameWriter.scala:257)
	at org.apache.spark.sql.DataFrameWriter.save(DataFrameWriter.scala:240)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.i